
# Niveau 1 : ML ou pas ML ?

## 1. Identifier le type de problème

Pour chaque situation choisissez parmi :
- classification ;
- régression ;
- clustering ;
- détection d'anomalies ;
- pas nécessairement du Machine Learning.

### Situation A
> Calculer automatiquement le montant de la taxe carbone d'une entreprise à partir du barème réglementaire officiel en vigueur.

### Situation B
> Prédire le niveau de concentration en particules fines (PM10) dans l'air pour la journée de demain dans une grande agglomération.

### Situation C
> Identifier automatiquement à partir d'images satellites si une parcelle forestière a subi une coupe rase illégale.

### Situation D
> Regrouper les communes françaises selon des profils de consommation d'eau potable et de taux de fuite sur leurs réseaux.

### Situation E
> Détecter qu'un capteur de niveau d'eau placé sur un fleuve renvoie soudainement une valeur physiquement impossible.

### Situation F
> Rechercher tous les documents administratifs contenant le mot-clé exact "biodiversité marine" dans une base d'archives.


## Correction du Niveau 1

| Situation | Réponse                  | Explication                                                                                              |
| --------- | ------------------------ | -------------------------------------------------------------------------------------------------------- |
| A         | Pas nécessairement du ML | Une règle réglementaire connue et exacte suffit.                                                         |
| B         | Régression               | La sortie attendue (concentration) est une valeur numérique continue.                                    |
| C         | Classification           | La sortie est binaire : "coupe rase" / "intacte".                                                        |
| D         | Clustering               | Les groupes ne sont pas définis à l'avance, on cherche à découvrir des structures.                       |
| E         | Détection d'anomalies    | On recherche des observations inhabituelles ou aberrantes par rapport au comportement normal du capteur. |
| F         | Pas nécessairement du ML | Une simple recherche textuelle déterministe suffit.                                                      |

> **Point important :** utiliser du Machine Learning simplement parce qu'il est disponible est une mauvaise pratique. Une règle simple, déterministe et fiable est souvent préférable lorsqu'elle suffit.

# 12. Une démarche de résolution

Une problématique de Machine Learning peut être abordée en plusieurs étapes.

```mermaid
graph TD
    A["Problème métier"] --> B{"Une règle simple suffit ?"}
    B -->|"Oui"| C(["Utiliser une solution déterministe"])
    B -->|"Non"| D{"Dispose-t-on de données ?"}
    D -->|"Non"| E(["Collecter des données ou revoir le problème"])
    D -->|"Oui"| F{"Dispose-t-on d'une cible ?"}
    F -->|"Oui"| G(["Apprentissage supervisé"])
    F -->|"Non"| H(["Apprentissage non supervisé ou détection d'anomalies"])

    classDef decision fill:#ffcc80,stroke:#e65100,stroke-width:2px,color:#000
    classDef resultOui fill:#a5d6a7,stroke:#2e7d32,stroke-width:2px,color:#000
    classDef resultNon fill:#ef9a9a,stroke:#c62828,stroke-width:2px,color:#000

    class B,D,F decision
    class C,G,H resultOui
    class E resultNon
```

Cette démarche est volontairement simplifiée, mais elle permet d'éviter une erreur fréquente :

> **Commencer par choisir un algorithme avant d'avoir correctement défini le problème.**


# Niveau 2 : Étude de cas (L'efficacité énergétique)

## 2. Étude de cas : les bâtiments publics

Le ministère dispose d'une base de données concernant plusieurs milliers de bâtiments publics.
Pour chaque bâtiment, on connaît :
- sa surface en m² ;
- son année de construction ;
- le nombre d'agents y travaillant ;
- sa localisation (département) ;
- les températures extérieures moyennes ;
- son type de chauffage (gaz, électricité, géothermie...) ;
- sa consommation énergétique annuelle réelle.

### Question 1
Une direction souhaite :
> **Prédire la consommation électrique exacte (en kWh) d'un bâtiment pour le trimestre prochain.**
Quel type de problème est-ce ?

### Question 2
Une direction souhaite :
> **Identifier automatiquement des groupes de bâtiments ayant des profils de déperdition thermique similaires pour cibler les campagnes de rénovation.**
Quel type de problème est-ce ?

### Question 3
Une direction souhaite :
> **Déterminer si un bâtiment est susceptible d'être classé comme "passoire thermique" (étiquette F ou G au DPE).**
Quel type de problème est-ce ?

### Question 4
La consommation d'un bâtiment suit les variations saisonnières habituelles, mais affiche soudainement une consommation nulle en plein hiver alors que le bâtiment est occupé.
Quel type de problème peut-on envisager pour repérer cela automatiquement à l'avenir ?


## Correction du Niveau 2

### Question 1
**Régression**, car la consommation à prédire (en kWh) est une valeur numérique continue.

### Question 2
**Clustering**, car les "profils de déperdition" ne sont pas connus à l'avance, l'algorithme doit regrouper les bâtiments qui se ressemblent.

### Question 3
**Classification**, car le résultat attendu est une catégorie (Passoire thermique : Oui / Non).

### Question 4
**Détection d'anomalies**, car on cherche à repérer un comportement qui s'écarte drastiquement du fonctionnement habituel normal, potentiellement dû à une panne de compteur.


# 14. Une première utilisation de scikit-learn

Dans la suite de la formation, nous utiliserons `scikit-learn`.

Une bonne pratique essentielle est de séparer les données destinées à l'apprentissage des données destinées à l'évaluation.

```mermaid
graph LR
    A["Jeu de données"] --> B["Train"]
    A --> C["Test"]
    B --> D["Apprentissage du modèle"]
    D --> E["Évaluation"]
    C --> E

    classDef input fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000
    classDef decision fill:#ffcc80,stroke:#e65100,stroke-width:2px,color:#000
    classDef resultOui fill:#a5d6a7,stroke:#2e7d32,stroke-width:2px,color:#000

    class A input
    class B,C decision
    class D decision
    class E resultOui
```

## Pourquoi ?

Si nous évaluons le modèle sur les données qu'il a utilisées pour apprendre, nous risquons d'obtenir une vision trop optimiste de ses performances.

L'objectif est de répondre à la question :

> **Le modèle fonctionne-t-il sur des données qu'il n'a jamais vues ?**


# 15. Le Data Leakage

Le **Data Leakage**, ou fuite de données, apparaît lorsqu'une information qui ne devrait pas être disponible au moment de la prédiction se retrouve indirectement utilisée par le modèle.

Exemple :

> Nous voulons prédire si un maison sera déclaré cher.

Mais nous construisons une variable à partir de la prix annuelle complète, alors que cette prix contient justement l'information permettant de déterminer si le maison est cher.

Le modèle dispose alors d'une information qu'il ne devrait pas connaître.


## Une règle fondamentale

> **Tout ce qui est appris à partir des données doit être appris uniquement sur le jeu d'entraînement.**

Cette règle concerne notamment :

- l'imputation des valeurs manquantes ;
- la normalisation ;
- la standardisation ;
- l'encodage ;
- la sélection de variables ;
- la réduction de dimension.

Dans `scikit-learn`, nous utiliserons donc systématiquement les `Pipeline` et `ColumnTransformer` lorsque cela est pertinent.


# 16. Exemple de Pipeline

Voici un exemple à exécuter pas à pas pour voir les grandes étapes d'un apprentissage machine.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. GÉNÉRATION DE DONNÉES FICTIVES (Pour les besoins du TP)
np.random.seed(42)
n_samples = 200

# Création d'un jeu de données simulant des prélèvements d'eau
data = pd.DataFrame({
    'ph': np.random.normal(7.2, 0.8, n_samples),
    'temperature_celsius': np.random.normal(15, 5, n_samples),
    'region': np.random.choice(['Bretagne', 'Occitanie', 'Grand_Est'], n_samples),
    'statut': np.random.choice(['Conforme', 'Polluée'], n_samples, p=[0.7, 0.3]) # 70% conforme
})



In [ ]:
print("--- 1. APERÇU DES DONNÉES BRUTES ---")
print(data.head(), "\n")


In [ ]:
# 2. SÉPARATION DES DONNÉES (Train / Test)
X = data[['ph', 'temperature_celsius', 'region']] # Features (Variables explicatives)
y = data['statut']                                # Target (Cible)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("--- 2. SÉPARATION DES DONNÉES ---")
print(f"Données d'entraînement : {X_train.shape[0]} prélèvements")
print(f"Données de test (inconnues du modèle) : {X_test.shape[0]} prélèvements\n")


In [ ]:
# 3. CRÉATION DU PIPELINE (Préparation + Modèle)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['ph', 'temperature_celsius']), # Mise à l'échelle des nombres
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['region']) # Encodage du texte
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression()) # Algorithme de classification
])


In [ ]:
# 4. APPRENTISSAGE (Entraînement sur X_train)
pipeline.fit(X_train, y_train)
print("--- 3. APPRENTISSAGE ---")
print("Le modèle a terminé son apprentissage sur les données d'entraînement.\n")



In [ ]:
# 5. PRÉDICTION SUR DE NOUVELLES DONNÉES (Le jeu de test)
predictions = pipeline.predict(X_test)

# Affichage des 5 premières prédictions face à la réalité
resultats_comparatifs = pd.DataFrame({
    'Valeur Réelle (y_test)': y_test.head(),
    'Prédiction du modèle': predictions[:5]
})

print("--- 4. PRÉDICTION SUR LE JEU DE TEST (Extraits) ---")
print(resultats_comparatifs)

L'intérêt est de regrouper dans un même objet :

```text
préparation des données
        +
transformation
        +
modèle
```

Cela limite notamment les risques de réaliser par erreur une transformation sur l'ensemble des données avant la séparation train/test.


# Niveau 3 : Challenge

# 17. Challenge : faut-il utiliser du Machine Learning ?

Vous êtes consultant en Data Science pour une entreprise.

Pour chaque situation, vous devez répondre à quatre questions :

1. **Le Machine Learning est-il pertinent ?**
2. **Quel type d'apprentissage ?**
3. **Quel type de problème ?**
4. **Quelles données seraient nécessaires ?**


## Cas 1 — Prévision de trafic

> Une organisation souhaite prévoir le nombre de véhicules circulant sur une portion d'autoroute dans les 30 prochaines minutes.

Réfléchissez notamment aux variables suivantes :

```text
heure
jour de la semaine
vacances scolaires
météo
historique du trafic
accidents
travaux
...
```


## Cas 2 — Surveillance de transactions

> Des milliers de transactions industriels transmettent des mesures toutes les cinq minutes. Le système doit signaler automatiquement les comportements inhabituels.

Questions :

- Avons-nous forcément besoin d'une cible ?
- Peut-on apprendre ce qu'est un comportement normal ?
- Quel problème cela évoque-t-il ?


## Cas 3 — Classification de documents

> Une administration reçoit automatiquement des milliers de documents. Elle souhaite les classer dans différentes catégories.

Exemples :

```text
urbanisme
transport
prix
biodiversité
eau
déchets
...
```

Questions :

- Comment obtenir la cible ?
- S'agit-il d'une classification ?
- Que se passe-t-il si seulement 5 % des documents sont annotés ?


## Cas 4 — Calcul réglementaire

> Une application doit déterminer le montant d'une taxe à partir de paramètres définis précisément par la réglementation.

Question :

> **Pourquoi serait-il probablement inutile d'utiliser du Machine Learning ?**


# Correction du challenge

## Cas 1 — Prévision de trafic

**Machine Learning : oui, potentiellement.**

Type :

> **Régression**, car la sortie est un nombre.

On dispose généralement de nombreuses données historiques permettant de construire un problème supervisé.


## Cas 2 — Surveillance de transactions

**Machine Learning : potentiellement oui.**

Une approche possible est la **détection d'anomalies**.

On peut notamment apprendre le comportement habituel des transactions puis détecter les observations qui s'en éloignent.

Une approche supervisée est également possible si l'on dispose d'un historique suffisamment riche d'incidents correctement étiquetés.


## Cas 3 — Classification de documents

**Machine Learning : oui.**

Il s'agit d'un problème de **classification supervisée** si les documents sont déjà étiquetés.

Si seulement une petite proportion des documents est annotée, on peut envisager :

- davantage d'annotation ;
- de l'apprentissage semi-supervisé ;
- d'autres techniques permettant d'exploiter les documents non annotés.


## Cas 4 — Calcul réglementaire

**Machine Learning : généralement non.**

Si la réglementation définit précisément le calcul, une fonction déterministe est préférable.

```text
Entrées
   ↓
Règle réglementaire
   ↓
Résultat
```

Il n'est pas pertinent de demander à un modèle statistique d'apprendre une règle que nous connaissons déjà exactement.


# 18. Les erreurs à éviter

## Erreur 1 — « Il y a beaucoup de données, donc il faut du ML »

Faux.

Une grande quantité de données ne signifie pas automatiquement qu'un modèle de Machine Learning est nécessaire.


## Erreur 2 — « Le Machine Learning trouve toujours la meilleure solution »

Faux.

Un modèle peut :

- apprendre des biais ;
- être trop complexe ;
- mal généraliser ;
- être difficile à expliquer ;
- nécessiter beaucoup de maintenance.


## Erreur 3 — Commencer par l'algorithme

Mauvaise démarche :

```text
J'utilise Random Forest.
Quel problème puis-je lui donner ?
```

Bonne démarche :

```text
Problème métier
      ↓
Données disponibles
      ↓
Type de problème
      ↓
Critère de succès
      ↓
Méthode appropriée
      ↓
Algorithme
```


## Erreur 4 — Confondre corrélation et causalité

Un modèle peut découvrir une relation statistique sans que cette relation soit causale.

Exemple simplifié :

> Les maisons consommant beaucoup d'prix sont souvent occupés par davantage de personnes.

Cela ne signifie pas nécessairement que le nombre d'agents est **la cause principale** de toutes les différences de prix.


# 19. Synthèse

À ce stade, vous devez retenir cinq idées.

### 1. Le Machine Learning apprend à partir de données

```text
Données → apprentissage → modèle → prédiction
```

### 2. Il existe plusieurs familles d'apprentissage

```text
Supervisé
Non supervisé
Semi-supervisé
```

### 3. Les problèmes supervisés les plus courants sont

```text
Classification → catégorie
Régression → nombre
```

### 4. Le non supervisé permet notamment

```text
Clustering → groupes
Détection d'anomalies → comportements inhabituels
```

### 5. Le ML n'est pas toujours nécessaire

> **Toujours commencer par le problème métier avant de choisir une technologie.**


# 20. Carte mentale finale

```mermaid
graph TD
    ML["Machine Learning"]

    ML --> SUP["Supervisé"]
    ML --> UNSUP["Non supervisé"]
    ML --> SEMI["Semi-supervisé"]

    SUP --> CLASS["Classification"]
    SUP --> REG["Régression"]

    UNSUP --> CLUST["Clustering"]
    UNSUP --> ANOM["Détection d'anomalies"]

    CLASS --> C1(["Catégorie"])
    REG --> R1(["Nombre"])
    CLUST --> C2(["Groupes"])
    ANOM --> C3(["Observations inhabituelles"])

    classDef decision fill:#ffcc80,stroke:#e65100,stroke-width:2px,color:#000
    classDef resultOui fill:#a5d6a7,stroke:#2e7d32,stroke-width:2px,color:#000
    classDef resultNon fill:#ef9a9a,stroke:#c62828,stroke-width:2px,color:#000

    class ML,SUP,UNSUP,SEMI,CLASS,REG,CLUST,ANOM decision
    class C1,R1,C2,C3 resultOui
```


# 21. Pour aller plus loin

La prochaine étape consiste à passer de la question :

> **« Quel type de problème avons-nous ? »**

à :

> **« Comment un algorithme apprend-il réellement à partir des données ? »**

Nous allons alors introduire progressivement :

- les jeux de données ;
- `X` et `y` ;
- l'entraînement et la prédiction ;
- la fonction de coût ;
- l'évaluation ;
- le surapprentissage (*overfitting*) ;
- la généralisation ;
- puis les premiers algorithmes de Machine Learning.


# À retenir

> **Le Machine Learning n'est pas une technologie que l'on applique indistinctement à tous les problèmes.**
>
> La première compétence d'un Data Scientist est de transformer une **problématique métier** en un **problème d'apprentissage correctement défini**.
